# 007_across_target3_cluster_selected_decode_visualize.ipynb

Across-condition clustering-first branch. This version targets exactly **3 clusters** for the primary clustering score, because the paper question is whether an archetype separates intact, word, and rest.

It still lets you change `CLUSTER_RANGE` if you want, but the default is `[3]`.


In [ ]:

ANALYSIS_TYPE = "temporal"   # "spatial" or "temporal"
LOAD_DIR = "msaa_flexible_outputs_npz"

K_VALUES = [5, 10, 25, 50, 75, 100]

# Selection mode:
#   "top_n"     -> keep the top N archetypes by clustering score
#   "threshold" -> keep all archetypes with clustering score >= threshold
SELECTION_MODE = "top_n"
TOP_N_CLUSTER = 5
CLUSTER_SCORE_THRESHOLD = 0.35

NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

# Clustering-score tuning
TARGET_N_CLUSTERS = 3
CLUSTER_RANGE = [3]  # target intact / word / rest
PURITY_WEIGHT = 1
BALANCE_WEIGHT = 1.0
PENALTY_MODE = "weak_sqrt"   # "sqrt", "weak_sqrt", "linear", or "none"

COND_COLORS = {"intact": "purple", "word": "green", "rest": "black"}

SCHAEFER_TXT = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt"
SCHAEFER_NII = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
POSTERIOR_MAT = "data/pieman/raw/pieman_posterior_K700.mat"

OUTPUT_DIR = f"007_across_target3_cluster_selected_decode_visualize_{ANALYSIS_TYPE}"

In [ ]:
from pathlib import Path

%matplotlib inline
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.spatial.distance import cdist
from sklearn.cluster import SpectralClustering
from matplotlib.colors import ListedColormap, to_rgba
from matplotlib import cm, colors as mcolors

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    import nibabel as nib
    from nilearn.input_data import NiftiMasker
    from nilearn import plotting as niplot
    NILEARN_AVAILABLE = True
except Exception:
    NILEARN_AVAILABLE = False
    print("nilearn / nibabel not available; brain plotting disabled.")

print("Ready.")
print("ANALYSIS_TYPE =", ANALYSIS_TYPE)

In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "007_across_target3_cluster_selected_decode_visualize"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"   # "pdf", "png", or "svg"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None
    _fig_counter += 1
    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)
    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
# ============================================================
# CACHE SETTINGS
# ============================================================

from pathlib import Path
import pickle

CACHE_DIR = Path("007_across_target3_cluster_selected_decode_visualize_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

USE_CACHE = True
OVERWRITE_CACHE = False

CLUSTER_SUMMARY_CACHE = CACHE_DIR / "cluster_summary_df.csv"
SELECTED_ARCHETYPES_CACHE = CACHE_DIR / "selected_archetypes_dict.npy"
PLOT_DATA_CACHE = CACHE_DIR / "plot_data_cache.pkl"

print("Cache directory:", CACHE_DIR.resolve())
print("USE_CACHE:", USE_CACHE)
print("OVERWRITE_CACHE:", OVERWRITE_CACHE)

In [ ]:

def to_float_array(x):
    return np.array(x, dtype=float)

def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()
    return {
        "K": int(data["K"]),
        "results_subj": results_subj,
        "condition_labels_str": np.array(data["condition_labels_str"].tolist()),
        "condition_codes": np.array(data["condition_codes"]) if "condition_codes" in data else None,
    }

loaded = {}
for K in K_VALUES:
    path = os.path.join(LOAD_DIR, f"{ANALYSIS_TYPE}AA_across_acrossCond_K{K}.npz")
    loaded[K] = load_msaa_npz(path)
    print("Loaded:", path)

In [ ]:

posterior = loadmat(POSTERIOR_MAT)
centers = np.asarray(posterior['posterior']['centers'][0][0][0][0][0], dtype=float)
widths = np.asarray(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T), dtype=float).ravel()

lookup_table = {
    'Vis': 'Visual',
    'SomMot': 'Somatomotor',
    'DorsAttn': 'Dorsal attention',
    'SalVentAttn': 'Ventral attention',
    'Limbic': 'Limbic',
    'Cont': 'Frontoparietal',
    'Default': 'Default mode'
}
network_colors = {
    'Visual': '#D7DF23',
    'Somatomotor': '#39B54A',
    'Dorsal attention': '#00A79D',
    'Ventral attention': '#27AAE1',
    'Limbic': '#1C75BC',
    'Frontoparietal': '#92278F',
    'Default mode': '#EE2A7B'
}
colors = ['#888888'] + [v for _, v in network_colors.items()]
network_codes = {k: i + 1 for i, k in enumerate(lookup_table.values())}

def nii2cmu(nifti_file, mask_file=None):
    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = nib.load(nifti_file) if type(nifti_file) == str else nifti_file
        mask = NiftiMasker(mask_strategy='background')
        mask.fit(nifti_file if mask_file is None else mask_file)

    S = img.get_sform()
    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]
    return {'Y': Y, 'R': R}

def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)
        label_weights = [sum(r[networks_cmu['Y'].ravel() == i]) for i in range(1, len(network_codes) + 1)]
        labels.append(np.argmax(label_weights) + 1)
    return pd.DataFrame({
        'code': labels,
        'Network': [list(lookup_table.values())[i - 1] for i in labels]
    })

if NILEARN_AVAILABLE:
    key = pd.read_csv(
        SCHAEFER_TXT,
        sep='\t',
        header=None,
        names=['id', 'name', 'x', 'y', 'z', 't']
    ).drop('t', axis=1)
    key['network'] = key['name'].apply(lambda x: lookup_table[x.split('_')[2]])
    key['code'] = key['network'].apply(lambda x: network_codes[x])
    key.set_index('id', inplace=True)
    key.loc[0, 'code'] = 0

    networks_cmu = nii2cmu(SCHAEFER_NII)
    networks_cmu['Y'] = np.atleast_2d(np.array([key.loc[i, 'code'] for i in networks_cmu['Y']]).astype(float))
    node_code_df = node_labels(centers, widths, networks_cmu)
else:
    node_code_df = None

In [ ]:

def decoder(corrs):
    out = pd.DataFrame({'rank':[0.0], 'accuracy':[0.0], 'error':[0.0]})
    T = corrs.shape[0]
    for t in range(T):
        decoded_ind = int(np.argmax(corrs[t, :]))
        out.loc[0, 'error'] += np.mean(np.abs(decoded_ind - t)) / T
        out.loc[0, 'accuracy'] += (decoded_ind == t)
        out.loc[0, 'rank'] += np.mean((corrs[t, :] <= corrs[t, t]).astype(int))
    out['error'] /= T
    out['accuracy'] /= T
    out['rank'] /= T
    return out

def get_xval_assignments(ndata, nfolds, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))
    for i in range(1, nfolds):
        inds = np.arange(i * groupsize, min((i + 1) * groupsize, ndata))
        group_assignments[inds] = i
    rng.shuffle(group_assignments)
    return group_assignments

def reconstruct_from_archetypes(sub, analysis_type, archetype_indices):
    sXC = to_float_array(sub["sXC"])
    S = to_float_array(sub["S"])
    idx = list(archetype_indices)

    if analysis_type == "spatial":
        Xhat = sXC[:, idx] @ S[idx, :]
    elif analysis_type == "temporal":
        Xhat = (sXC[:, idx] @ S[idx, :]).T
    else:
        raise ValueError("analysis_type must be 'spatial' or 'temporal'")

    Xhat = (Xhat - Xhat.mean(axis=0, keepdims=True)) / (Xhat.std(axis=0, keepdims=True) + 1e-8)
    return Xhat

def build_recon_stack(subjects, analysis_type, archetype_indices):
    return np.stack([reconstruct_from_archetypes(sub, analysis_type, archetype_indices) for sub in subjects], axis=0)

def run_timepoint_decoding(recon_stack, nfolds=2, nreps=20, seed=42):
    N, T, V = recon_stack.shape
    rng = np.random.default_rng(seed)
    all_results = []
    for rep in range(nreps):
        fold_ids = get_xval_assignments(N, nfolds, rng=rng)
        for i in range(nfolds):
            in_mask = (fold_ids == i)
            out_mask = ~in_mask
            in_mean = recon_stack[in_mask].mean(axis=0)
            out_mean = recon_stack[out_mask].mean(axis=0)
            corrs = 1.0 - cdist(in_mean, out_mean, metric='correlation')
            res = decoder(corrs)
            res["rep"] = rep
            res["fold"] = i
            all_results.append(res)
    return pd.concat(all_results, ignore_index=True)

def summarize_decoding_by_condition(results_subj, condition_labels_str, analysis_type, archetype_indices, nfolds=2, nreps=20, seed=42):
    rows = []
    for cond_name in np.unique(condition_labels_str):
        idx = np.where(condition_labels_str == cond_name)[0]
        sub_cond = [results_subj[i] for i in idx]
        recon_stack = build_recon_stack(sub_cond, analysis_type, archetype_indices)
        dec_df = run_timepoint_decoding(recon_stack, nfolds=nfolds, nreps=nreps, seed=seed)
        rows.append({
            "condition": cond_name,
            "mean": float(dec_df["accuracy"].mean()),
            "sem": float(dec_df["accuracy"].std() / np.sqrt(max(len(dec_df), 1)))
        })
    return pd.DataFrame(rows)

In [ ]:

def purity_score(y_cluster, y_class):
    y_cluster = np.asarray(y_cluster)
    y_class = np.asarray(y_class)
    total = 0
    for c in np.unique(y_cluster):
        mask = (y_cluster == c)
        _, counts = np.unique(y_class[mask], return_counts=True)
        total += counts.max()
    return total / len(y_cluster)

def equal_size_balance(labels):
    labels = np.asarray(labels)
    _, counts = np.unique(labels, return_counts=True)
    N = counts.sum()
    k = len(counts)
    ideal = N / k
    imbalance = np.sum(np.abs(counts - ideal)) / (2 * N)
    return float(1.0 - imbalance)

def cluster_count_penalty(n_clusters, mode="sqrt"):
    if mode == "linear":
        return 1.0 / n_clusters
    elif mode == "sqrt":
        return 1.0 / np.sqrt(n_clusters)
    elif mode == "weak_sqrt":
        return 1.0 / (n_clusters ** 0.25)
    elif mode == "none":
        return 1.0
    else:
        raise ValueError("mode must be 'linear', 'sqrt', 'weak_sqrt', or 'none'")

def score_clustering_solution(labels, cond, purity_weight=1.5, balance_weight=1.0, penalty_mode="sqrt"):
    labels = np.asarray(labels)
    cond = np.asarray(cond)
    n_clusters = len(np.unique(labels))
    purity = purity_score(labels, cond)
    balance = equal_size_balance(labels)
    penalty = cluster_count_penalty(n_clusters, mode=penalty_mode)
    score = (purity ** purity_weight) * (balance ** balance_weight) * penalty
    _, counts = np.unique(labels, return_counts=True)
    return {
        "n_clusters": n_clusters,
        "purity": purity,
        "balance": balance,
        "score": score,
        "cluster_sizes": counts.tolist(),
    }

def get_clustering_subject_matrix(results_subj, analysis_type, k, eps=1e-8):
    Xk = np.stack([to_float_array(sub["sXC"])[:, k] for sub in results_subj], axis=0)
    mu = Xk.mean(axis=1, keepdims=True)
    sd = Xk.std(axis=1, keepdims=True)
    Xk = (Xk - mu) / (sd + eps)
    return np.nan_to_num(Xk, nan=0.0, posinf=0.0, neginf=0.0)

def scan_cluster_numbers(Xk, cond_codes, cluster_range=range(2, 9), purity_weight=1.5, balance_weight=1.0, penalty_mode="sqrt"):
    sim = np.corrcoef(Xk)
    sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)
    sim = np.clip(sim, -1.0, 1.0)
    np.fill_diagonal(sim, 1.0)

    rows = []
    for n_clusters in cluster_range:
        aff = (sim + 1.0) / 2.0
        np.fill_diagonal(aff, 1.0)
        labels = SpectralClustering(
            n_clusters=n_clusters,
            affinity="precomputed",
            assign_labels="kmeans",
            random_state=0
        ).fit_predict(aff) + 1

        scored = score_clustering_solution(
            labels, cond_codes,
            purity_weight=PURITY_WEIGHT,
            balance_weight=BALANCE_WEIGHT,
            penalty_mode=PENALTY_MODE
        )
        rows.append({
            "n_clusters": n_clusters,
            **scored,
        })
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

In [ ]:

def select_archetypes(dfK, mode="top_n", top_n=5, threshold=0.35):
    if mode == "top_n":
        return dfK.sort_values("cluster_score", ascending=False).head(top_n)["archetype"].astype(int).tolist()
    elif mode == "threshold":
        selected = dfK[dfK["cluster_score"] >= threshold]["archetype"].astype(int).tolist()
        if len(selected) == 0:
            selected = [int(dfK.sort_values("cluster_score", ascending=False).iloc[0]["archetype"])]
        return selected
    else:
        raise ValueError("SELECTION_MODE must be 'top_n' or 'threshold'")

In [ ]:

all_cluster_rows = []
selected_decode_rows = []
selected_archetypes_dict = {}

for K in K_VALUES:
    cur = loaded[K]
    results_subj = cur["results_subj"]
    condition_labels_str = cur["condition_labels_str"]
    cond_codes = cur["condition_codes"]

    n_archetypes = np.asarray(results_subj[0]["sXC"]).shape[1]
    rows = []

    print(f"\n===== K={K} | n_archetypes={n_archetypes} =====")

    for k in range(n_archetypes):
        Xk = get_clustering_subject_matrix(results_subj, ANALYSIS_TYPE, k)
        scan_df = scan_cluster_numbers(
            Xk, cond_codes,
            cluster_range=CLUSTER_RANGE,
            purity_weight=PURITY_WEIGHT,
            balance_weight=BALANCE_WEIGHT,
            penalty_mode=PENALTY_MODE
        )
        best_cluster = scan_df.iloc[0]
        rows.append({
            "K": K,
            "archetype": k,
            "best_n_clusters": int(best_cluster["n_clusters"]),
            "purity": float(best_cluster["purity"]),
            "balance": float(best_cluster["balance"]),
            "cluster_score": float(best_cluster["score"]),
            "cluster_sizes": best_cluster["cluster_sizes"],
        })

    dfK = pd.DataFrame(rows).sort_values("cluster_score", ascending=False).reset_index(drop=True)
    all_cluster_rows.append(dfK)

    selected_arches = select_archetypes(
        dfK,
        mode=SELECTION_MODE,
        top_n=TOP_N_CLUSTER,
        threshold=CLUSTER_SCORE_THRESHOLD
    )
    selected_archetypes_dict[K] = selected_arches

    dec_summary = summarize_decoding_by_condition(
        results_subj,
        condition_labels_str,
        ANALYSIS_TYPE,
        selected_arches,
        nfolds=NFOLDS_DECODE,
        nreps=NREPS_DECODE,
        seed=RNG_SEED
    )
    dec_summary["K"] = K
    dec_summary["selected_archetypes"] = str(selected_arches)
    dec_summary["n_selected"] = len(selected_arches)
    selected_decode_rows.append(dec_summary)

    print("Selected archetypes:", selected_arches)
    display(dfK.head(10))
    display(dec_summary)

cluster_scores_df = pd.concat(all_cluster_rows, ignore_index=True)
selected_decode_df = pd.concat(selected_decode_rows, ignore_index=True)


## Decode performance of clustering-selected archetypes


In [ ]:

plt.figure(figsize=(8, 5))
for cond_name in ["intact", "word", "rest"]:
    sub = selected_decode_df[selected_decode_df["condition"] == cond_name].sort_values("K")
    plt.errorbar(
        sub["K"], sub["mean"], yerr=sub["sem"],
        marker="o", capsize=4,
        color=COND_COLORS.get(cond_name, "black"),
        label=cond_name
    )
plt.xlabel("K")
plt.ylabel("Decoding accuracy")
plt.title(f"{ANALYSIS_TYPE} AA across | decoding from clustering-selected archetypes")
plt.legend()
plt.tight_layout()
save_current_fig()
plt.show()
plt.close()


## How many archetypes were selected at each `K`


In [ ]:

nsel_df = selected_decode_df[["K", "n_selected"]].drop_duplicates().sort_values("K")
plt.figure(figsize=(7, 4))
plt.plot(nsel_df["K"], nsel_df["n_selected"], marker="o")
plt.xlabel("K")
plt.ylabel("Number selected")
plt.title(f"{ANALYSIS_TYPE} AA across | number of clustering-selected archetypes")
plt.tight_layout()
save_current_fig()
plt.show()
plt.close()
display(nsel_df)


## Visualize the selected archetypes


In [ ]:

def plot_spatial_timecourse(results_subj, k, K):
    Xk = np.stack([to_float_array(sub["sXC"])[:, k] for sub in results_subj], axis=0)
    mean = Xk.mean(axis=0)
    sem = Xk.std(axis=0) / np.sqrt(max(Xk.shape[0], 1))
    x = np.arange(len(mean))
    plt.figure(figsize=(9, 4))
    plt.plot(x, mean, color="black")
    plt.fill_between(x, mean - sem, mean + sem, color="gray", alpha=0.2)
    plt.title(f"K={K} | archetype {k} | temporal motif")
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()
def plot_temporal_coeff_timecourse(results_subj, condition_labels_str, k, K):
    rows = []
    for i, sub in enumerate(results_subj):
        coeff = to_float_array(sub["S"])[k, :]
        rows.append(pd.DataFrame({
            "time": np.arange(len(coeff)),
            "value": coeff,
            "condition": condition_labels_str[i]
        }))
    df = pd.concat(rows, ignore_index=True)
    plt.figure(figsize=(9, 4))
    for cond_name, color in COND_COLORS.items():
        cur = df[df["condition"] == cond_name]
        if len(cur) == 0:
            continue
        summ = cur.groupby("time")["value"].agg(["mean", "std", "count"]).reset_index()
        summ["sem"] = summ["std"] / np.sqrt(summ["count"].clip(lower=1))
        plt.plot(summ["time"], summ["mean"], color=color, label=cond_name)
        plt.fill_between(summ["time"], summ["mean"] - summ["sem"], summ["mean"] + summ["sem"], color=color, alpha=0.2)
    plt.title(f"K={K} | archetype {k} | coefficient timecourse")
    plt.legend()
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()
def coefficient_to_alpha(weights, keep_mask, alpha_min=0.15, alpha_max=1.0):
    w = np.asarray(weights, dtype=float).ravel()
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    wk = w[keep_mask]
    if len(wk) == 0:
        return np.array([])
    wmin, wmax = wk.min(), wk.max()
    if np.isclose(wmax, wmin):
        return np.full(len(wk), alpha_max)
    scaled = (wk - wmin) / (wmax - wmin)
    return alpha_min + (alpha_max - alpha_min) * scaled

def plot_spatial_coeff_brain(results_subj, k, K, thr_frac=0.30):
    if not NILEARN_AVAILABLE:
        return None, None, None
    coeffs = np.stack([to_float_array(sub["S"])[k, :] for sub in results_subj], axis=0)
    weights = np.nan_to_num(coeffs.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)
    wmax = np.max(weights)
    if wmax <= 0:
        return None, None, None
    keep = weights >= (thr_frac * wmax)
    centers_sel = centers[keep]
    node_codes_local = node_labels(centers, widths, networks_cmu)
    codes_sel = node_codes_local.loc[keep, 'code'].to_numpy()
    node_colors = [colors[i] for i in codes_sel]
    disp = niplot.plot_connectome(
        np.eye(centers_sel.shape[0]),
        centers_sel,
        node_size=10,
        node_color=node_colors,
        display_mode="lyrz",
        title=f"K={K} | archetype {k} | coefficient brain plot"
    )
    save_current_fig()
    plt.show()
    plt.close()
    return disp, node_codes_local, keep

def plot_temporal_spatial_map(results_subj, k, K):
    if not NILEARN_AVAILABLE:
        return
    vals = np.stack([to_float_array(sub["sXC"])[:, k] for sub in results_subj], axis=0).mean(axis=0)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    vmax = np.max(np.abs(vals))
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    cmap = cm.get_cmap("coolwarm")
    node_colors = [cmap(norm(v)) for v in vals]
    node_sizes = 8 + 18 * (np.abs(vals) / (vmax + 1e-8))
    disp = niplot.plot_connectome(
        np.eye(centers.shape[0]),
        centers,
        node_color=node_colors,
        node_size=node_sizes,
        display_mode="lyrz",
        title=f"K={K} | archetype {k} | signed spatial motif map"
    )
    save_current_fig()
    plt.show()
    plt.close()
    return disp

def plot_network_pie_for_selected_nodes(node_codes_local, keep_mask, title="Network composition"):
    selected = node_codes_local.loc[keep_mask].copy()
    counts = selected["Network"].value_counts().reindex(list(network_colors.keys()), fill_value=0)
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    pie_colors = [network_colors[name] for name in counts.index]
    ax.pie(
        counts.values,
        labels=counts.index,
        colors=pie_colors,
        autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
        startangle=90,
        counterclock=False
    )
    ax.set_title(title)
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()
    plt.close(fig)

In [ ]:

for K in K_VALUES:
    print("\n" + "="*90)
    print(f"K={K} | selected archetypes = {selected_archetypes_dict[K]}")
    print("="*90)

    cur = loaded[K]
    results_subj = cur["results_subj"]
    condition_labels_str = cur["condition_labels_str"]

    for k in selected_archetypes_dict[K]:
        if ANALYSIS_TYPE == "spatial":
            plot_spatial_timecourse(results_subj, k, K)
            disp, node_codes_local, keep_mask = plot_spatial_coeff_brain(results_subj, k, K, thr_frac=0.30)
            if disp is not None:
                try:
                    disp.close()
                except Exception:
                    pass
            if node_codes_local is not None and keep_mask is not None:
                plot_network_pie_for_selected_nodes(
                    node_codes_local, keep_mask,
                    title=f"K={K} | archetype {k} | selected-node network composition"
                )

        elif ANALYSIS_TYPE == "temporal":
            plot_temporal_coeff_timecourse(results_subj, condition_labels_str, k, K)
            disp = plot_temporal_spatial_map(results_subj, k, K)
            if disp is not None:
                try:
                    disp.close()
                except Exception:
                    pass

In [ ]:
# ============================================================
# Clustered similarity heatmaps WITH condition-colored labels
# ============================================================

# Define condition colors (adjust if needed)
COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black"
}


def reorder_by_labels_and_within_similarity(sim, labels):
    labels = np.asarray(labels)
    unique_labels = np.unique(labels)
    order = []

    for lab in unique_labels:
        idx = np.where(labels == lab)[0]
        sub_sim = sim[np.ix_(idx, idx)]
        sub_order = idx[np.argsort(-sub_sim.mean(axis=1))]
        order.extend(sub_order.tolist())

    order = np.array(order)
    labels_sorted = labels[order]
    boundaries = np.where(labels_sorted[1:] != labels_sorted[:-1])[0] + 1

    return order, labels_sorted, boundaries


def best_cluster_solution_for_archetype(results_subj, cond_codes, k):
    Xk = get_clustering_subject_matrix(results_subj, ANALYSIS_TYPE, k)

    sim = np.corrcoef(Xk)
    sim = np.nan_to_num(sim)
    sim = np.clip(sim, -1, 1)
    np.fill_diagonal(sim, 1.0)

    best = None
    rows = []

    for n_clusters in CLUSTER_RANGE:
        aff = (sim + 1) / 2
        np.fill_diagonal(aff, 1.0)

        labels = SpectralClustering(
            n_clusters=n_clusters,
            affinity="precomputed",
            assign_labels="kmeans",
            random_state=0
        ).fit_predict(aff) + 1

        scored = score_clustering_solution(
            labels,
            cond_codes,
            purity_weight=PURITY_WEIGHT,
            balance_weight=BALANCE_WEIGHT,
            penalty_mode=PENALTY_MODE
        )

        rows.append({
            "n_clusters": n_clusters,
            "purity": scored["purity"],
            "balance": scored["balance"],
            "score": scored["score"]
        })

        if best is None or scored["score"] > best["score"]:
            best = {
                "labels": labels,
                "sim": sim,
                "n_clusters": n_clusters,
                **scored
            }

    order, labels_sorted, boundaries = reorder_by_labels_and_within_similarity(
        best["sim"],
        best["labels"]
    )

    best["sim_sorted"] = best["sim"][np.ix_(order, order)]
    best["order"] = order
    best["labels_sorted"] = labels_sorted
    best["boundaries"] = boundaries
    best["scan_df"] = pd.DataFrame(rows).sort_values("score", ascending=False)

    return best


def plot_cluster_heatmap(results_subj, cond_labels, cond_codes, k, K, show_subject_ids=True):
    res = best_cluster_solution_for_archetype(results_subj, cond_codes, k)

    order = res["order"]
    cond_sorted = np.array(cond_labels)[order]

    # label indices
    if show_subject_ids:
        tick_labels = [str(i) for i in order]
    else:
        tick_labels = [""] * len(order)

    plt.figure(figsize=(7, 6))
    ax = sns.heatmap(
        res["sim_sorted"],
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        xticklabels=tick_labels,
        yticklabels=tick_labels
    )

    # COLOR LABELS BY CONDITION
    for tick_label, cond in zip(ax.get_xticklabels(), cond_sorted):
        tick_label.set_color(COND_COLORS.get(cond, "gray"))
        tick_label.set_rotation(90)
        tick_label.set_fontsize(8)

    for tick_label, cond in zip(ax.get_yticklabels(), cond_sorted):
        tick_label.set_color(COND_COLORS.get(cond, "gray"))
        tick_label.set_fontsize(8)

    # draw cluster boundaries
    for b in res["boundaries"]:
        ax.axhline(b, color="black", linewidth=2)
        ax.axvline(b, color="black", linewidth=2)

    plt.title(
        f"{ANALYSIS_TYPE} AA | K={K} | archetype {k}\n"
        f"clusters={res['n_clusters']} | purity={res['purity']:.2f} | "
        f"balance={res['balance']:.2f} | score={res['score']:.3f}"
    )

    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()
    display(res["scan_df"])

    return res


# ============================================================
# RUN FOR SELECTED ARCHETYPES
# ============================================================

selected_cluster_results = {}

for K in K_VALUES:
    cur = loaded[K]
    results_subj = cur["results_subj"]
    cond_labels = cur["condition_labels_str"]
    cond_codes = cur["condition_codes"]

    selected_cluster_results[K] = {}

    print("\n" + "="*80)
    print(f"K={K} | selected archetypes: {selected_archetypes_dict[K]}")
    print("="*80)

    for k in selected_archetypes_dict[K]:
        selected_cluster_results[K][k] = plot_cluster_heatmap(
            results_subj,
            cond_labels,
            cond_codes,
            k,
            K,
            show_subject_ids=True
        )

In [ ]:

cluster_scores_df.to_csv(
    os.path.join(OUTPUT_DIR, f"cluster_scores_all_archetypes_{ANALYSIS_TYPE}_across.csv"),
    index=False
)
selected_decode_df.to_csv(
    os.path.join(OUTPUT_DIR, f"decode_from_cluster_selected_archetypes_{ANALYSIS_TYPE}_across.csv"),
    index=False
)
np.save(
    os.path.join(OUTPUT_DIR, f"selected_archetypes_{ANALYSIS_TYPE}_across.npy"),
    selected_archetypes_dict,
    allow_pickle=True
)

print("Saved outputs to:", OUTPUT_DIR)